# 07 · Bounds, Constraints, and Slip Bases

*Encode physical admissibility while recognizing the bias it can introduce.*

**Learning objectives**

- distinguish quadratic regularization from hard constraints
- apply non-negative, bounded, and linear-inequality constraints
- derive fixed-rake, azimuth, and plate-motion bases
- recognize constraint-induced artifacts

**Prerequisites:** Chapter 04; recommended: Chapter 05  
**Estimated time:** 60–90 minutes

Notation follows the [glossary](../docs/glossary.md); axes, units, signs, and
ordering follow [GeoDef conventions](../docs/conventions.md).


## Motivation

A quadratic penalty can make slip smooth or small, but it cannot say “this
thrust does not slip normally” or “coupling stays between zero and one.” Bounds,
inequalities, and reduced slip bases encode such information exactly. Exact
constraints buy admissibility at the cost of bias when the premise is wrong.


## Theory deepening: feasible sets and reduced bases

NNLS restricts the solution to the non-negative orthant; active parameters at
zero define the boundary of the feasible set. General bounds form a box, while
$Cm\le d$ cuts it with half-spaces and yields a quadratic program.

For fixed rake $\rho$, one amplitude $a_k$ maps into two physical components:

$$s_{s,k}=a_k\cos\rho,\qquad s_{d,k}=a_k\sin\rho.$$

This reduction is exact, not a soft preference. Geographic azimuth accounts
for patch strike; a plate basis rotates into plate-parallel and perpendicular
coordinates. Constraints can pile slip against edges or bounds, so active
constraints and residual structure must be reported alongside the solution.


## Priors that regularization can't express

Regularization penalizes *how much* a model is disliked (a quadratic cost).
Some priors are **hard** and one-sided:

- slip should not reverse sense (no back-slip): $\mathbf m \ge 0$;
- slip magnitude is capped: $\mathbf m \le u$;
- more general geologic limits: $C\mathbf m \le \mathbf d_{\text{ineq}}$.

`geodef.invert.solve()` handles these through the `bounds` and `constraints` arguments
and picks the right solver automatically:

- `bounds=(0, None)` → **non-negative least squares (NNLS)**;
- `bounds=(lb, ub)` → **bounded least squares**;
- `method='constrained', constraints=(C, d)` → a **quadratic program (QP)**
  with $C\mathbf m \le \mathbf d$.

Bounds may be a scalar, a per-component pair, or a full per-parameter array.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.matrix(
    fault, geodef.data.gnss(lon=glon, lat=glat, east=_zero, north=_zero, up=_zero, sigma_east=_one, sigma_north=_one, sigma_up=_one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.data.gnss(
    lon=glon, lat=glat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_sta, sigma), sigma_north=np.full(n_sta, sigma), sigma_up=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

## Non-negative slip

A lightly-smoothed unconstrained inversion dips slightly *negative* at the edges
— spurious back-slip. Requiring $\mathbf m \ge 0$ removes it. Because the true
slip is one-signed, this is a physically correct prior.

In [ ]:
kw = dict(components="dip", regularization="laplacian", regularization_strength=0.2)
free = geodef.solve(fault, gnss, **kw)                    # can go negative
nn = geodef.solve(fault, gnss, bounds=(0, None), **kw)    # auto-NNLS

print(f"unconstrained min slip: {free.dip_slip.min()*100:+.1f} cm (spurious back-slip)")
print(f"non-negative  min slip: {nn.dip_slip.min()*100:+.1f} cm")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
lim = abs(free.dip_slip).max()
geodef.plot.slip(fault, free.dip_slip, ax=axes[0], cmap="RdBu_r",
                 vmin=-lim, vmax=lim, title="Unconstrained", colorbar_label="Slip (m)")
geodef.plot.slip(fault, nn.dip_slip, ax=axes[1], cmap="RdBu_r",
                 vmin=-lim, vmax=lim, title="Non-negative (NNLS)", colorbar_label="Slip (m)")
plt.tight_layout()

## Bounded least squares

Bounds can be two-sided. Cap the slip at 1 m to see where the upper bound starts
to bite (and bias the fit).

In [ ]:
capped = geodef.solve(fault, gnss, bounds=(0, 1.0), **kw)
print(f"max slip, uncapped: {nn.dip_slip.max():.2f} m")
print(f"max slip, capped:   {capped.dip_slip.max():.2f} m (hits the 1.0 m ceiling)")
print(f"reduced chi^2, uncapped {nn.reduced_chi2:.2f}  vs capped {capped.reduced_chi2:.2f}")

## Fixed slip direction

Another way to encode a prior is to fix the *direction* of slip and solve for one
amplitude per patch. `components='rake'` fixes a single rake in each patch's
local strike-dip frame; `components='azimuth'` fixes a geographic slip azimuth
(correct even when strike varies). This halves the unknowns and cleanly enforces
a sense of motion.

In [ ]:
# rake = 90 deg is pure dip-slip in the local frame (matches our true slip).
rk = geodef.solve(fault, gnss, components="rake", rake=90.0,
                   regularization="laplacian", regularization_strength=1.0)
print(f"components='rake' solves {rk.dip_slip.size} amplitudes "
      f"(vs {2*N} for 'both')")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vmax = slip_true[N:].max()
geodef.plot.slip(fault, slip_true[N:], ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="True", colorbar_label="Slip (m)")
geodef.plot.slip(fault, rk.dip_slip, ax=axes[1], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="Fixed-rake amplitude",
                 colorbar_label="Slip (m)")
plt.tight_layout()

## Checkpoint questions

**What is the difference between a bound and regularization?**

<details><summary>Answer</summary>



A bound excludes models; regularization ranks all models by a continuous penalty.



</details>

**Why can positivity create edge concentrations?**

<details><summary>Answer</summary>



When negative compensation is forbidden, the optimizer can move amplitude to weakly constrained feasible patches.



</details>

**When is azimuth preferable to rake?**

<details><summary>Answer</summary>



When patch strike varies and one geographic direction should stay fixed.



</details>


## Common mistakes

- Applying positivity to the wrong signed component can enforce the opposite mechanism.
- A parameter pinned at a bound has a non-Gaussian, asymmetric uncertainty.
- Using one rake on a curved mesh can imply different geographic directions.


## Recap

- Bounds and inequalities define admissible model sets.
- Reduced bases encode direction assumptions exactly and reduce dimension.
- Constraints can stabilize estimates while introducing visible bias and artifacts.

Return to the [workflow decision guides](../docs/workflow.md) when adapting
this method. The course map in [the tutorial README](README.md) identifies the
next chapter and optional branches.


## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are published separately in `solutions/07_bounds_and_constraints_solutions.ipynb`.

1. Compare WLS, NNLS, and fixed-rake solutions and fits.
2. Tighten an upper bound until it changes reduced chi-squared substantially.
3. Apply per-component bounds to a two-component solve.
4. Challenge: build the fixed-rake reduction matrix and compare its prediction with `components='rake'`.


## Further reading

- Lawson & Hanson (1995), non-negative least squares.
- Nocedal & Wright (2006), constrained optimization.
- Segall (2010), physically constrained geodetic inversion.
